# 들어가기 전에... 
## kinase가 뭐죠? 
Protein kinase(이하 PK)는 단백질 인산화 효소입니다. 인산? 그건 뭔데요? Phosphate라고 하는건데 화학식은 PO4(3가 음이온)입니다. 유기체에게 있어서 인산은 굉장히 중요한 요소입니다. DNA와 RNA의 골격을 이루고 있는 것도 인산이고, ATP의 trihosphate는 인산기가 세 개라는 의미이고(PK는 보통 여기서 뗴다 답니다), 뼈와 인지질(세포막을 구성합니다)을 구성하고 있죠. 

PK는 종류가 정말 다양하고, 우리 체내에서 다양한 신호를 전달하는 중요한 일을 하고 있습니다. 여러분이 먹는 밥에서 나오는 포도당도 Hexokinase가 인산화를 해 주는 게 해당과정의 첫빠따죠. 여러분이 먹는 것, 대사하는 것, 숨쉬는 것, 내보내는 것, 세포분열... 여러가지 과정에 관여하는 게 인산화 효소, kinase입니다. 

당연하게도, 이 신호가 개박살나면 병납니다. 

## 그럼 Inhibitor는 뭔데요? 
인히비터는 말 그대로 뭘 방해하는 애라는 뜻입니다. 그러니까 Kinase inhibitor는 '인산화 효소를 방해하는' 녀석이라는 얘기가 되겠죠? 

## 프로젝트 정보
- 인원: 1인
- 파이썬 버전: 3.10
- 데이터 리소스: ChEMBL
- 필요한 모듈: Numpy, Pandas, Matplotlib, Seaborn

In [ ]:
# 모! 듈! 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats # 통계분석용
from statsmodels.stats.multicomp import pairwise_tukeyhsd # 어노바 친구 튜키

# 그래프 기본 테마 설정
sns.set_theme(palette="Blues", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Griun Gellyroll' # 제가... 픽셀체 이런거 좋아해서... 
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈 
plt.rcParams['axes.unicode_minus'] = False

# 내겐 너무나도 험난했던 불러오기
- 아니... 구분자 해줬잖아... 왜 오류나 왜... 

In [ ]:
df = pd.read_csv('data/Kinase_inhibitor.csv', sep=';', quotechar='"', engine='python', on_bad_lines='skip') # 저거 안해주면 에러나요
df.shape

## 정보 확인

### df.info()

In [ ]:
df.info()

### df.describe()

In [ ]:
df.describe()

In [ ]:
df.describe(include='O')

### df.isna().sum()

In [ ]:
df.isna().sum() # 오 주여... 

### .head()

In [ ]:
df.head()

### df.columns

In [ ]:
df.columns

# 전처리
- 일단 크게 Max phase(임상 상태)랑 분자량 보고 할 듯 합니다. 

## Max phase에 따른 구분
|Max phase|상태|
|--|--|
|4|승인됨|
|3~0|임상|
|-1|엎음|

In [ ]:
df['Max Phase'].value_counts()

In [ ]:
df['Status'] = df['Max Phase'].map({4: 'Approved', 3: 'Clinical', 2: 'Clinical', 1: 'Clinical', 0: 'Clinical', -1: 'Failed'}) 

In [ ]:
df['Status'].value_counts()

- 저기 엎은거 왜 이렇게 많은겨...

## Type에 따른 분류
- 중분류

|Type|카테고리|
|--|--|
|Small molecules|동일|
|나머지|Other|

- 세분류

|Type|카테고리|
|--|--|
|Small molecules|동일|
|unknown|동일|
|Protein, Antibody, Antibody drug conjugate, Enzyme, Vaccine component|Biologics|
|Oligosaccharide|Sugars|
|Oligonucleotide, Gene|Nucleic acids|

### 중분류

In [ ]:
df['Type'].value_counts()

In [ ]:
# 중분류
df['Category_1'] = df['Type'].apply(lambda x: "Small molecule" if x == "Small molecule" else "Other")

In [ ]:
df['Category_1'].value_counts()

### 세분류
- 이건 람다 못씁니다. 딕셔너리 만들어서 세분화해야됨... 

In [ ]:
# 그룹 정의
groups = {
    'Biologics': ['Protein', 'Antibody', 'Antibody drug conjugate', 'Enzyme', 'Vaccine component'],
    'Nucleic acids': ['Oligonucleotide', 'Gene'],
    'Sugars': ['Oligosaccharide'],
    'Small molecule': ['Small molecule'],
    'Unknown': ['Unknown']
}

# 근데 뒤집을거면 처음부터 반대로 만들면 안되는거임? 
# 안된답니다. 
mapping = {val: key for key, values in groups.items() for val in values}

# 예 적용됐습니다. 
df['Category_2'] = df['Type'].map(mapping)

In [ ]:
df['Category_2'].value_counts()

## 분자량으로 나누기
- 의미가 있나? 생각해보니 항체나 단백질같은 건 당연히 500보다 클텐데. 

In [ ]:
df['Moecular_classification'] = df['Molecular Weight'].apply(lambda x: 'Heavy' if x > 500 else 'Light')

# 분석 드가자!
## 임상 단계별로 보기
### 임상 단계별 평균

In [ ]:
df.groupby('Status')['Molecular Weight'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(df, x = 'Status', y = 'Molecular Weight', hue = 'Status')
plt.yscale('log')
plt.title('Molecular Weight by Status')
plt.show()

In [ ]:
df.query('`Molecular Weight` > 6000')

- 쟤 올리고뉴클레오타이드라 원래 뚠뚠~합니다. 습성 황반변성(wet AMD) 치료를 위해 개발된 세계 최초의 소간섭 RNA(siRNA) 기반 혈관내피성장인자(VEGF) 억제제입니다. 
- 저놈의 타겟은 VEGF라는 놈인데, 얘는 PK가 아니라 PK를 활성화하는 스위치입니다. 그니까 자동차로 치자면 자동차(VEGFR)는 있고 열쇠(VEGF)를 안줘서 시동을 못 걸게 하는거죠. 

#### 근데 RNA 백신도 있잖아요. 
그냥 들어가면 개발살나니까 이런 방법을 씁니다. 
- **갑옷 입히기 (Chemical Modification):** RNA의 등뼈(Backbone)를 화학적으로 바꿔서 RNase 가위가 안 먹게 만듭니다.
- **배달 가방 (LNP, Lipid Nanoparticle):** 기름 막으로 약을 감싸서 RNase의 눈을 속이고 세포 안까지 안전하게 모셔다줍니다. 
- **주소 부착 (GalNAc):** 아예 간세포 같은 특정 목표물로 바로 가게끔 이정표를 달아버립니다. ~~이거 퀵 아니냐~~

In [ ]:
df.groupby(['Status', 'Category_2'])['Molecular Weight'].mean() # 평균

- 대체로 핵산 > 당 > 생체분자 > 저분자 순으로 크다. 
- 근데 핵산은 승인된게 하나도 없네? 

### 생리활성

In [ ]:
df.groupby('Status')['Bioactivities'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(df, x = 'Status', y = 'Bioactivities', hue = 'Status')
plt.yscale('log')
plt.title('Bioactivities by Status')
plt.show()

- Failed가 대체로 활성이 낮은데, 이거 혹시... 들어가자마자 개발살나서 그런가? 

In [ ]:
df.query('Bioactivities > 3500')[['Name','Status','Bioactivities']]

- 생리활성이 3500을 넘어가는 분자들은 전부 저분자들이었다. (MOLIBRESIB은 임상, 나머지는 승인)
- 저 임상중인 친구 찾아보니까 부작용이 거의 뭐 초기 항암제 수준이더만... 
- Imatinib이 낯이 익으시다고요? 혹시 필라델피아 유전자에 대해 아십니까? 

In [ ]:
df.groupby(['Status', 'Category_2'])['Bioactivities'].mean() # 평균

### AlogP

In [ ]:
df.groupby('Status')['AlogP'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(df, x = 'Status', y = 'AlogP', hue = 'Status')
plt.title('AlogP by Status')
plt.show()

### 통계분석
-저 셋이 다 다른지 봅시다. 근데 어노바가 될라나 모르겠네... 
#### ANOVA

In [ ]:
# 표본 수 확인용
# 결측값 안지우면 통계량 안나와요... 
df_approved = df.query('Status == "Approved"')['Bioactivities'].dropna()
df_clinical = df.query('Status == "Clinical"')['Bioactivities'].dropna()
df_failed = df.query('Status == "Failed"')['Bioactivities'].dropna()

- 표본 수 충분한데? 

In [ ]:
f_stats, p_val = stats.f_oneway(df_approved, df_clinical, df_failed)
print(f'F-Statistics: {f_stats:.2f}')

if p_val < 0.05:
    print(f"p-value: {p_val:.2e}: 귀무가설 기각! Tukey 드가자!") # 아 Tucky가 아니구나... 튜키씨 미안... 
else:
    print(f"p-value: {p_val:.2e}: 귀무가설 기각 실패")

#### Tukey HSD
- 튜키씨 미안... 나 님 스펠링 아직도 헷갈려... (Tucky 아니고 Tukey임)

In [ ]:
all_vals = list(df_approved) + list(df_clinical) + list(df_failed)
all_labels = (['Approved'] * len(df_approved) + ['Clinical'] * len(df_clinical) + ['Failed'] * len(df_failed))

tukey = pairwise_tukeyhsd(endog=all_vals, groups=all_labels, alpha=0.05)
print(tukey)

- Approved와 Clinical, Approved와 Failed간에 유의미한 차이가 있다는 얘깁니다. 

### AlogP가 6보다 큰 애들
- 그니까 얘들은 무지 기름지다... 이 얘기다. 

In [ ]:
# 임상 단계별로 봐야 한다... 
# 실패한 애들부터 보자. 
df.query('AlogP > 6 and Status == "Failed"')[['Name','Status','Type', 'AlogP','Bioactivities']]

In [ ]:
# 임상중
df.query('AlogP > 6 and Status == "Clinical"')[['Name','Status','Type', 'AlogP','Bioactivities']]

In [ ]:
# 임상중, AlogP > 9
df.query('AlogP > 9 and Status == "Clinical"')[['Name','Status','Type', 'AlogP','Bioactivities','Max Phase']]

- SONROTOCLAX 저친구는 타겟이 되는 단백질의 위치가 기름진 곳이라 "뭐? 기름? 오히려 좋아!"상태인겁니다. 간독성에 대해서는 투약 용량을 조절하거나 다른 방법을 쓴다면 승인될 수도 있는거죠. 
- 그 외에는 대부분 '너무 기름져서' 2상인 케이스라고 보시면 되겠습니다. 

In [ ]:
# 승인
df.query('AlogP > 6 and Status == "Approved"')[['Name','Status','Type', 'AlogP','Bioactivities']]

### AlogP가 0보다 작은 애들 

In [ ]:
df.query('AlogP < 0 and Status == "Failed"')[['Name','Status','Type', 'AlogP','Bioactivities']]

- 쟤 CDS가 어떻게 됨? 

In [ ]:
df.query('AlogP < 0 and Status == "Clinical"')[['Name','Status','Type', 'AlogP','Max Phase', 'Bioactivities']]

In [ ]:
# 2상 아닌 애들
df.query('AlogP < 0 and Status == "Clinical" and `Max Phase`!= 2')[['Name','Status','Type', 'AlogP','Max Phase','Bioactivities']]

- MOLNUPIRAVIR, OBELDESIVIR: 코로나바이러스용 항바이러스제입니다. 이건 DNA 염기 비슷하게 생겼는데, 바이러스가 이 짝퉁 염기를 갖고 증식에 썼다가 아... 짭이었어... 설계도 망했어... 아... 죽었어... 이렇게 되는겁니다. 
- NACUBACTAM, ZIDEBACTAM: 항생제입니다. 여러분, 약을 꼭 입으로 먹어야 한다는 법은 없어요. 쟤들은 혈관에 다이렉트로 꽂아서 균이 있는 곳까지 직배송하는 애들입니다. 
- CROCIN: 사프란에서 추출되는 당류입니다. 타입이 Oligosaccharide예요. 당이라 원래 그런겁니다. 

In [ ]:
df.query('AlogP < 0 and Status == "Approved"')[['Name','Status','Type', 'AlogP','Type','Bioactivities']]

- EPTIFIBATIDE: 뱀독 유래 합성 펩타이드입니다. 심장 혈전 및 심근경색을 예방하는 정맥 투여용 당단백질 IIb/IIIa 억제제로 사용합니다. 

### 부작용 플래그가... 있나?

In [ ]:
df.query('`Withdrawn Flag` != False')[['Name','Status','Type', 'AlogP','Type','Withdrawn Flag']]

~~아니 플래그가 왜이렇게 많아 이거~~
1. CERIVASTATIN: 횡문근융해증... 고지혈증 고치다가 신장 투석 받게 생긴겁니다. 
2. LUMIRACOXIB: 간독성
3. VALDECOXIB: 심혈관 질환(심장마비, 뇌졸중)의 위험성 증가 및 심각한 피부 반응 위험
4. XIMELAGATRAN: 와파린(항응고제)의 대체제였는데 아... 간독성 아... 
5. TOLRESTAT: 간독성. 미국에서 승인이 안 났던 건 탈리도마이드랑 비슷하네요. 
6. UMBRALISIB: 어... 그... 항암제인데요... 다른 의미로 효과가 있었습니다... (infections, neutropenia, diarrhea and non-infectious colitis, hepatotoxicity, and severe cutaneous reactions)
7. NOMIFENSINE MALEATE: 항우울제인데... 아... 
    - 급성 용혈성 빈혈: 내 몸의 면역계가 갑자기 내 적혈구를 적으로 오해해서 다 터뜨려버립니다. (피가 모자라!)
    - 노미펜신 열(Fever): 약만 먹으면 고열이 펄펄 끓습니다.
    - 심각한 간 독성: 간 수치가 수직 상승합니다.
    - 조증/환각: 정신과 약(항우울제)인데, 우울증 고치려다 사람이 너무 들떠서 조증이 오거나 환각을 봅니다.
8. TOLCAPONE: 간독성
9. APROTININ: 피를 너무 잘 멈춰서... 아... 

## 임상 단계별 #RO5 Violations
- 이건 분자들만 볼 예정입니다. 다른거는 원래 뚠뚠한 애들이라 당연히 어김... 

In [ ]:
df.query('Category_1 == "Small molecule"').groupby('Status')['#RO5 Violations'].size()

In [ ]:
df.query('Category_1 == "Small molecule"').groupby(['Status', '#RO5 Violations']).size().unstack()


### RO5 삼진아웃
#### 승인됨

In [ ]:
df[df['#RO5 Violations'] > 2].query('Type == "Small molecule" and Status == "Approved"')[['Name','Status','Molecular Weight','HBA','HBD','AlogP','#RO5 Violations']]

- 분자량, HBA, HBD가 RO5 위반이다. 

#### 임상 진행중

In [ ]:
df[df['#RO5 Violations'] > 2].query('Type == "Small molecule" and Status == "Clinical"')[['Name','Status','Molecular Weight','HBA','HBD','AlogP','#RO5 Violations']]

#### 엎음

In [ ]:
df[df['#RO5 Violations'] > 2].query('Type == "Small molecule" and Status == "Failed"')[['Name','Status','Molecular Weight','HBA','HBD','AlogP','#RO5 Violations']]

- HBD는 좋았는데... 일단 너무 뚱땡했음...
- 전체적으로 임상중인 애들 중에 RO5를 위반하는 애들이 많다. 

### 언노운 쟤 뭐 하는 애임? 

In [ ]:
df.query('Category_2 == "Unknown"').groupby('Status').size()

In [ ]:
df.query('Category_2 == "Unknown" and Status == "Approved"')[['Name','Synonyms']]

In [ ]:
df.query('Category_2 == "Unknown" and Status == "Failed"')[['Name','Synonyms']]

In [ ]:
df.query('Category_2 == "Unknown" and Status == "Clinical"')[['Name','Synonyms']]

## Gene? 이거 유전자?

In [ ]:
df.query('Type == "Gene"')[['Name','Synonyms','Status']]

- 저친구 찾아보니까 바이러스인데??
- 바이러스인데 써도 되나 괜찮으시죠? 저친구는 감염을 일으키는 친구가 아니라 암세포 찾아가서 핑 찍게 개조(?)한 겁니다. 